In [1]:
import numpy as np
from numba.core.types import none
from numba.np.arrayobj import np_arange

from main_functions import *
import matplotlib.pyplot as plt

recording_path = r"data\2026-02-25_mb_fish1_rec2"
save_path = r"results\2026-02-25_mb_fish1_rec2"


In [2]:
fluorescence, recording, phase, ca_rec_group_id_fun = digest_folder(recording_path, plane=0)
process_recording(recording, phase, radial_bin_num=16)


data\2026-02-25_mb_fish1_rec2\suite2p\plane0
Calculate frame timing of Fluorescence
Load ROI data
Estimated imaging rate 2.00Hz
Add ROI stats and signals
Add display phase data


100%|██████████| 109/109 [00:00<00:00, 132.14it/s]


Pick from motion vectors matrix to form cmn_motion_vectors_3d


100%|██████████| 105/105 [00:00<00:00, 175.33it/s]


In [3]:
Dff_all_neuron = np.load(os.path.join(save_path, "deconvolved_Dff_original.npy"))
Dff_resampled = scipy.interpolate.interp1d(recording['ca_times'], Dff_all_neuron, kind='nearest')(
        recording['time_resampled'])

In [4]:
# sample first neuron from loop (over all neurons)
i=0
test_neuron_dff = Dff_resampled[i, :]

# einfacher als mit GMMs (wie's in 2019 paper is), robust
detect_events_with_derivative(recording, test_neuron_dff)

In [6]:
Dff_all_neuron.shape

(480, 6617)

# calculate reverse correlation function

In [5]:
# ETA berechnung, Abb mit cnts für bewegungsrichtugen im vf
bootstrap_num=1000

# ROI data
test_neuron_signal_selection = recording['signal_selection'] # timepoints mask giving events
# Recording data
sample_rate = recording['sample_rate']
radial_bin_edges = recording['radial_bin_edges']
cmn_phase_selection = recording['cmn_phase_selection'] # timepoints mask giving CMN stim
motion_vectors_2d = recording['cmn_motion_vectors_2d']

motion_vectors_2d_filtered = motion_vectors_2d[test_neuron_signal_selection, :, :]
radial_bin_norms, radial_bin_etas = calculate_local_directions(motion_vectors_2d_filtered, radial_bin_edges)

recording[f'radial_bin_etas'] = radial_bin_etas # ETA's`
recording[f'radial_bin_norms'] = radial_bin_norms

min_frame_shift = 4 * sample_rate
max_frame_shift = int(cmn_phase_selection.sum() - min_frame_shift)
frame_shifts = np.random.randint(min_frame_shift, max_frame_shift, size=(bootstrap_num))

signal_within_cmn_selection = test_neuron_signal_selection[cmn_phase_selection]
signal_indices = signal_within_cmn_selection.nonzero()[0][:,None]


(1000,)

In [8]:
idcs=np.mod(signal_indices + frame_shifts, signal_within_cmn_selection.size).T
idcs.shape, idcs[0].shape

((1000, 6631), (6631,))

In [82]:
mv=motion_vectors_2d[cmn_phase_selection]

angles=np.arctan2(mv[:,:,0], mv[:,:,1])
velocities=np.linalg.norm(mv, axis=2)
mv.shape, angles.shape, velocities.shape

((18000, 320, 2), (18000, 320), (18000, 320))

In [83]:
vel=velocities[idcs[0]]
ang=angles[idcs[0]]
bins=radial_bin_edges
vel.shape, ang.shape, bins.shape

((6631, 320), (6631, 320), (17,))

In [19]:
bins[:-1].shape, ang[:, :, None].shape, vel[:, :, None].shape

((16,), (6631, 320, 1), (6631, 320, 1))

In [59]:
mv=mv[idcs[0]]

In [84]:
# ang=np.arctan2(mv[:,:,0], mv[:,:,1])
# vel=np.linalg.norm(mv, axis=2)
# np.mean(vel[:, :, None] * np.logical_and(bins[:-1] <= ang[:, :, None], ang[:, :, None] <= bins[1:]), axis=0)

In [91]:
norms = vel[:,  :, None] * np.logical_and(bins[:-1] <= ang[:,  :, None], ang[:, :, None] <= bins[1:])
etas=norms.mean(axis=0)

In [92]:
etas.shape

(320, 16)

In [93]:
etas

array([[0.02884057, 0.03106635, 0.03383759, ..., 0.03014117, 0.02804294,
        0.03094191],
       [0.028403  , 0.02929452, 0.03149781, ..., 0.03009683, 0.02982529,
        0.03056187],
       [0.0269034 , 0.03095217, 0.02948871, ..., 0.0358162 , 0.02870081,
        0.02935995],
       ...,
       [0.02734283, 0.02489254, 0.0345475 , ..., 0.0278352 , 0.02864951,
        0.03404342],
       [0.03391748, 0.03156216, 0.03847021, ..., 0.02813849, 0.02645274,
        0.02607751],
       [0.02967805, 0.02627039, 0.03931066, ..., 0.03010832, 0.02765241,
        0.0293674 ]], shape=(320, 16))

In [ ]:
# radial_bin_bs_etas = np.zeros((bootstrap_num,) + radial_bin_etas.shape)
# bins = radial_bin_bs_etas
# for s in tqdm(range(bootstrap_num)):
#         # Circularly permutate signal
#         ### perm_signal_selection = np.roll(signal_within_cmn_selection, frame_shifts[s])
#
#         # Get motion vectors of permutated signal
#         bs_motion_vectors = motion_vectors_2d[cmn_phase_selection][perm_signal_selection]
#
#         # Calculate vector ETAs for each local radial bin
          # REPLACED BELOW
#         #radial_bin_bs_etas[s] = calculate_local_directions(bs_motion_vectors, radial_bin_edges)[1]
#
#         motion_angles = np.arctan2(bs_motion_vectors[:, :, 1], bs_motion_vectors[:, :, 0])
#         motion_velocities = np.linalg.norm(bs_motion_vectors, axis=2)
#         from numpy import newaxis
#         bin_norms = motion_velocities[:, :, newaxis] * np.logical_and(radial_bin_edges[:-1] <= motion_angles[:, :, newaxis], motion_angles[:, :, newaxis] <= radial_bin_edges[1:])
#         radial_bin_bs_etas[s] = np.mean(bin_norms, axis=0)
#
# recording[f'radial_bin_bs_etas'] = radial_bin_bs_etas

### functions

In [ ]:
# def calculate_local_directions(motvecs: np.ndarray, bin_edges: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
#     # Convert to angle and velocity
#     motion_angles = np.arctan2(motvecs[:, :, 1], motvecs[:, :, 0])
#     motion_velocities = np.linalg.norm(motvecs, axis=2)
#     # Backwards transform (keep for reference):
#     # motion_vectors_2d = np.zeros((*motion_angles.shape[:2], 2))
#     # motion_vectors_2d[:,:,0] = motion_velocities * np.cos(motion_angles)
#     # motion_vectors_2d[:,:,1] = motion_velocities * np.sin(motion_angles)
#
#     # Calculate bin vectors for each patch and frame weighted by the local velocities
#     bin_norms = motion_velocities[:, :, None] * np.logical_and(bin_edges[:-1] <= motion_angles[:, :, None],
#                                                                motion_angles[:, :, None] <= bin_edges[1:])
#
#     # Calculate ETAs
#     bin_etas = np.mean(bin_norms, axis=0)
#
#     return bin_norms, bin_etas

# compare results to original pipeline results

In [73]:
import pickle

In [169]:
with open('radial_bin_etas.npy', 'rb') as f:
    radial_bin_etas = np.load(f)


with open('radial_bin_etas_slow.pkl', 'rb') as f:
    radial_bin_etas_slow = pickle.load(f)

etas, etas_ = radial_bin_etas, radial_bin_etas_slow
etas.shape, etas_.shape

((1024, 320, 16), (1024, 320, 16))

In [177]:
error=np.sum(radial_bin_etas - radial_bin_etas_slow)
error/ radial_bin_etas.sum() , error/radial_bin_etas_slow.sum()

(np.float64(-0.0001114989680933777), np.float64(-0.00011148653248694494))

# run full pipeline for one neuron

In [ ]:
# einfacher als mit GMMs (wie's in 2019 paper is), robust
detect_events_with_derivative(recording, test_neuron_dff)

# Calcium-event-triggered-averages berechnung
calculate_reverse_correlations_shm(recording, bootstrap_num=1024)

# 1. Teil Bootstrap test, bootstrap verteilung berechnungen, welche revcorr significant
calculate_directional_significance(recording)

# added by me: 2. Teil
calculate_directional_preference(recording)

# clusteranalyse
find_clusters(recording)

# cluster based statistics, (2-step NP bootsrap test)
calculate_cluster_significances(recording)

plot_rf_overview(recording, i, save_path)

In [9]:
f'test{str(1)}'

'test1'

# inspect recording dict

In [12]:
save_path

'results\\2026-02-25_mb_fish1_rec2'

In [19]:
import pickle
with open(os.path.join(recording_path, 'recording_dicts', f'recording_neuron{str(i)}.pkl'), 'rb') as f:
    recording = pickle.load(f)

In [27]:
recording.keys()

dict_keys(['animal_id', 'rec_date', 'rec_id', 'roi_num', 'signal_length', 'imaging_rate', 'ca_times', 'record_group_ids', 'time_resampled', 'sample_rate', 'display/attrs/__vxpy_status', 'display/attrs/__vxpy_version', 'display/CMN3D20240606Vel140Scale7Long/centers_0', 'display/CMN3D20240606Vel140Scale7Long/indices_0', 'display/CMN3D20240606Vel140Scale7Long/local_motion_vectors_0', 'display/CMN3D20240606Vel140Scale7Long/motion_vectors_0', 'display/CMN3D20240606Vel140Scale7Long/rotation_quats_0', 'display/CMN3D20240606Vel140Scale7Long/vertices_0', 'display/__record_group_id', 'display/__time', 'display/protocol0/__protocol_module', 'display/protocol0/__protocol_name', 'display/protocol0/__start_record_group_id', 'display/protocol0/__start_time', 'display/protocol0/__target_phase_count', 'display/protocol0/__target_repeat_interval_ids', 'positions', 'patch_corners', 'patch_indices', 'cmn_phase_selection_original', 'cmn_phase_selection', 'cmn_motion_vectors_3d', 'cmn_motion_vectors_2d', 'c

In [31]:
recording['radial_bin_bs_etas'].shape, recording['radial_bin_bs_etas'].shape

((1024, 320, 16), (1024, 320, 16))

In [41]:
p=recording['radial_bin_p_values']

In [53]:
plt.hist(p.flatten(), bins=200);
np.sum(p<.05), p.size

(np.int64(2088), 5120)

In [62]:
np.any(p>.05, axis=1).sum()

np.int64(320)

In [66]:
recording['radial_bin_significances']

array([[ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0., -1.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ..., -1.,  0.,  1.],
       ...,
       [ 1.,  1., -1., ...,  0.,  1.,  1.],
       [ 1.,  1.,  0., ...,  0.,  1.,  1.],
       [ 1.,  1., -1., ...,  0.,  1.,  1.]], shape=(320, 16))

# fitting optic flow field

scipy minimize, dont care about values, since only angles count.
define rotation axis: azimuth angle, elevation angle



In [102]:
radial_bin_centers=recording['radial_bin_centers']
radial_bin_centers

array([-2.94524311, -2.55254403, -2.15984495, -1.76714587, -1.37444679,
       -0.9817477 , -0.58904862, -0.19634954,  0.19634954,  0.58904862,
        0.9817477 ,  1.37444679,  1.76714587,  2.15984495,  2.55254403,
        2.94524311])

In [103]:
recording['radial_bin_bs_etas'].nbytes/1024**2

20.0

In [104]:
positions=recording['positions']
positions.shape

(320, 3)

In [105]:
fig=plt.figure()
ax=fig.add_subplot(projection='3d')
ax.scatter(recording['positions'][:,0],recording['positions'][:,1],recording['positions'][:,2])
ax.scatter(0,0,0)
plt.show()

In [99]:
import scipy

In [ ]:
def TOF(angle_azimuth, angle_elevation):
    """
        Translational optic flow field.
        Returns a translational optic flow field for a given translation axis.

        Parameters:
            angle_azimuth (float):
                angle in radians of the azimuth of the translation axis.
            angle_elevation (float):
                angle in radians of the elevation of the translation axis.
            positions (np.array):
                3D positions to sample flow field at.

        Returns:
            F (np.array):
                Return values of optic flow field (3D vector) at each point
                given in positions.
            FoC, FoE (np.array):
                Focus of Expansion and Focus of Contraction of the optic flow field.
                Given as 3D unit vector.
    """


In [101]:
positions.shape

(320, 3)

In [ ]:
def RSSangle(F, E):
